# WTI Crude Oil — Adaptive Agent: Self-Directed Study (Notebook 5 of 7)

> **Part 5 of 7.** Builds on the stateless backtest in [`04_systematic_backtest_eval.ipynb`](04_systematic_backtest_eval.ipynb).

Every method in Notebook 4 was **stateless** — configured once, run the same way each time.  
This notebook introduces an agent that is different: it can **learn from experience**.

The paradigm shift: instead of configuring a model, we onboard an analyst.  
We give the analyst a task, historical data, and a set of tools.  
The analyst explores the data, draws conclusions, and decides whether to update  
its own forecasting strategy — governed by evidence rules in its `meta-learning` skill.

**What this notebook produces:**

| Strategy dir | Contents |
|---|---|
| `wti-strategy/` | Clean initial state — never modified |
| `wti-strategy-trained/` | Strategy after one self-directed study session |

---
## 0. Setup

In [ ]:
import warnings
from pathlib import Path

from aieng.forecasting.methods.agentic import build_adk_agent
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig
from energy_oil_forecasting.adaptive_agent import build_wti_adaptive_config


warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
_NB_DIR = Path(".")
_SKILLS_ROOT = _NB_DIR / "adaptive_agent" / "skills"
_CURRICULUM_DIR = _NB_DIR / "adaptive_agent" / "curriculum"

# Clean seed — read-only baseline, never written to by training.
SEED_STRATEGY_DIR = _SKILLS_ROOT / "wti-strategy"
# Strategy state after the self-directed study session.
TRAINED_STRATEGY_DIR = _SKILLS_ROOT / "wti-strategy-trained"

# ── Model ─────────────────────────────────────────────────────────────────────
AGENT_MODEL = "gemini-3.5-flash"

# ── Run guards ────────────────────────────────────────────────────────────────
# Expensive by default; outputs are committed after first run.
# Set RUN_STUDY = True only to regenerate from scratch.
# Set RESEED = True to reset the trained strategy to the clean seed before running.
RUN_STUDY = False  # Self-directed study session (live API calls)
RESEED = False  # Reset wti-strategy-trained/ to clean seed first

print("Setup complete.")
print(f"  Seed:    {SEED_STRATEGY_DIR}")
print(f"  Trained: {TRAINED_STRATEGY_DIR}")

---
## 1. Before — The Agent's Starting State

The seed strategy (`wti-strategy/`) contains domain priors: a sensible initial  
approach, but no evidence-backed calibration corrections.  
It is the same strategy the **untrained agent** uses in Notebook 6.

The trained variant starts from an identical copy of this seed.  
Set `RESEED = True` in Setup if you want to reset it before a fresh study run.

In [ ]:
if RESEED:
    import shutil  # noqa: PLC0415

    from aieng.forecasting.methods.agentic.adaptive_skill import AdaptiveSkillStore  # noqa: PLC0415
    from energy_oil_forecasting.adaptive_agent.skill_state import WtiStrategyState  # noqa: PLC0415

    TRAINED_STRATEGY_DIR.mkdir(exist_ok=True)
    shutil.copy2(SEED_STRATEGY_DIR / "skill_state.yaml", TRAINED_STRATEGY_DIR / "skill_state.yaml")
    store = AdaptiveSkillStore(skill_dir=TRAINED_STRATEGY_DIR, state_type=WtiStrategyState)
    store.save(store.load())
    print("wti-strategy-trained/ reset to clean seed.")
else:
    print("RESEED = False — keeping existing wti-strategy-trained/ state.")

print()
print("Initial strategy (wti-strategy/SKILL.md):")
print("─" * 60)
print((SEED_STRATEGY_DIR / "SKILL.md").read_text())

---
## 2. Self-Directed Study

We give the agent one open-ended analytical task: explore 2025 WTI price data  
and assess whether its current forecasting approach is well-calibrated.

The agent has access to:
- `fetch-yfinance` — live price data from Yahoo Finance (with temporal cutoffs)
- `vol-regime` — volatility regime classification
- `trend-projection` — trend fitting and interval calibration
- `meta-learning` — evidence governance rules for updating strategy
- Strategy mutation tools — to record observations, open hypotheses, and apply corrections

The agent decides what to compute, what conclusions to draw, and whether any  
finding clears the evidence bar for updating its `wti-strategy-trained/` skill.

> **Run guard:** `RUN_STUDY = False` by default — the trained strategy state  
> is committed so this notebook runs reproducibly without live API calls.

In [ ]:
_STUDY_PROMPT = (
    "You have access to historical WTI crude oil price data via run_code. "
    "Please do the following:\n\n"
    "1. Fetch the daily WTI close price series for the full year 2025 using "
    'yfinance (ticker: CL=F, end="2026-01-01").\n'
    "2. Compute 21-day rolling realized volatility. Classify each day into a "
    "vol regime using the thresholds in your vol-regime skill.\n"
    "3. Simulate the errors a simple trend-projection forecaster would make "
    "at 5, 10, and 21 business-day horizons during each regime. Approximate "
    "this using the historical return distribution within each regime window.\n"
    "4. Summarize: in which regimes and at which horizons does trend-projection "
    "tend to produce the largest errors? Is there a directional bias?\n\n"
    "Based on your analysis, decide whether any findings meet the evidence "
    "threshold in your meta-learning skill. If they do, record them using "
    "the appropriate mutation tools. If not, explain what additional evidence "
    "you would need before updating your strategy."
)

if RUN_STUDY:
    config = build_wti_adaptive_config(model=AGENT_MODEL, strategy_dir=TRAINED_STRATEGY_DIR)
    agent = build_adk_agent(config)
    runner = AdkTextRunner(
        agent,
        config=AdkTextRunnerConfig(
            app_name="wti_self_directed_study",
            enable_langfuse_tracing=True,
            langfuse_tags=["energy-oil", "adaptive-agent", "self-directed-study"],
            langfuse_trace_name="wti-adaptive-self-directed-study",
        ),
    )
    print("Running self-directed study session...")
    print("(Live API calls + E2B sandbox — may take several minutes.)\n")
    reply = await runner.run_text_async(_STUDY_PROMPT)
    (_CURRICULUM_DIR / "study_response.txt").write_text(reply, encoding="utf-8")
    print(reply)
else:
    _f = _CURRICULUM_DIR / "study_response.txt"
    if _f.exists():
        print(_f.read_text())
    else:
        print("[Study session not yet run. Set RUN_STUDY = True and re-run.]")

---
## 3. After — What the Agent Learned

The cell below shows the trained strategy state.  
Look at what changed relative to the clean seed:

- **Observations**: patterns the agent noticed during analysis
- **Hypotheses**: candidate corrections it opened for future confirmation
- **Calibration corrections**: confirmed adjustments now applied at inference
- **Approach narrative**: how the agent describes its own strategy in its own words

These are the changes that will be active when the agent makes predictions  
in Notebook 6.

In [ ]:
import yaml  # noqa: PLC0415


def _load_state(d: Path) -> dict:
    return yaml.safe_load((d / "skill_state.yaml").read_text())


seed_state = _load_state(SEED_STRATEGY_DIR)
trained_state = _load_state(TRAINED_STRATEGY_DIR)

print("What changed after self-directed study:")
print("─" * 60)
for key, label in [
    ("observations", "Observations"),
    ("hypotheses", "Hypotheses"),
    ("calibration_corrections", "Calibration corrections"),
]:
    before = len(seed_state.get(key, []))
    after = len(trained_state.get(key, []))
    delta = f"+{after - before}" if after >= before else str(after - before)
    print(f"  {label:28s}: {before} → {after}  ({delta})")

approach_changed = trained_state.get("approach_narrative", "") != seed_state.get("approach_narrative", "")
print(f"  {'Approach narrative':28s}: {'UPDATED' if approach_changed else "unchanged'"}")

In [ ]:
print("Trained strategy (wti-strategy-trained/SKILL.md):")
print("─" * 60)
print((TRAINED_STRATEGY_DIR / "SKILL.md").read_text())

---
## Next: Protected Evaluation

Notebook 6 evaluates both the **untrained agent** (uses `wti-strategy/`)  
and the **trained agent** (uses `wti-strategy-trained/`) on the 2026 eval spec —  
a period of significant market volatility the agent has never seen.

The eval is deliberately **frozen**: the agent cannot update its strategy  
during evaluation, so the comparison is a clean before/after of what  
the self-directed study session contributed.